# Interaktivni action-conditioned world model — demo

Pritisneš smjer → model generiše sledeća 4 latentna frejma (16 piksel-frejmova) → nastavlja odatle.

**Kernel: `Python (minWM - torch 2.9.1+cu128)`** — provjeri gore desno prije pokretanja.

Model se učitava **jednom** (ćelija 2, ~5 min). Poslije toga svaki klik košta samo sampling (~2–5 s).

**Ograničenje, otvoreno rečeno:** model je treniran teacher-forcing-om — uvijek je vidio *pravi* kontekst.
U rollout-u jede sopstveni izlaz, pa greška raste. Kratki nizovi (2–4 bloka) drže, duži degradiraju.
To je razlog zašto minWM ima Stage 2/3 (self-forcing) poslije faze koju mi koristimo.

In [ ]:
import os, sys, time
os.environ.setdefault('USER','mls10'); os.environ.setdefault('LOGNAME','mls10')
os.environ.setdefault('HOME','/home/mls10')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
REPO='/home/mls10/minWM-dawidzard'
sys.path[:0]=[f'{REPO}/Wan21', f'{REPO}/shared', f'{REPO}/lora_action']
os.chdir(REPO)

import lmdb, numpy as np, torch
from omegaconf import OmegaConf
from PIL import Image
import ipywidgets as W
from IPython.display import display, clear_output
from wan_utils.lmdb_ import get_array_shape_from_lmdb, retrieve_row_from_lmdb
from train_lora_action import ActionEncoderV2

CHECKPOINT = '/home/mls10/checkpoints/bair_lora_big/step_8000.pt'
LMDB_PATH  = '/tmp/bair_lmdb/test'     # held-out split
N_STEPS    = 24                        # 4 with a DMD checkpoint -> ~6x faster
D          = 0.07                      # hard range of the real displacement dims
# smjer verifikovan na sirovim BAIR pikselima (bez modela):
#   dim0 < 0 -> DESNO na slici,  dim0 > 0 -> LIJEVO
#   dim1 < 0 -> GORE,            dim1 > 0 -> DOLJE
DIRS = {'up':(0.0,-D), 'down':(0.0,+D), 'right':(-D,0.0), 'left':(+D,0.0), 'still':(0.0,0.0)}
print('imports OK')

In [ ]:
# ==== UČITAVANJE MODELA — traje ~5 min, pokreni SAMO JEDNOM ====
device = torch.device('cuda'); torch.set_grad_enabled(False)
t0 = time.time()

ckpt = torch.load(CHECKPOINT, map_location='cpu'); rank = ckpt['args']['rank']
config = OmegaConf.load('Wan21/configs/ar_camera_tf.yaml')
config = OmegaConf.merge(OmegaConf.load('Wan21/configs/default_config.yaml'), config)
from model import CameraCausalDiffusion
model = CameraCausalDiffusion(config, device=device)

base = torch.load('/tmp/local_ckpts/Wan21/Action2V/ar_diffusion_tf/model.pt', map_location='cpu')
gen_sd = base.get('generator_ema', base.get('generator'))
try:
    model.generator.load_state_dict(gen_sd)
except RuntimeError:
    model.generator.load_state_dict({k.replace('model._fsdp_wrapped_module.','model.',1):v
                                     for k,v in gen_sd.items()}, strict=False)
for m in (model.generator, model.text_encoder, model.vae):
    m.to(device=device, dtype=torch.bfloat16)

from peft import LoraConfig, inject_adapter_in_model
inject_adapter_in_model(LoraConfig(r=rank, lora_alpha=rank*2,
                                   target_modules=['q','k','v','ffn.0','ffn.2']), model.generator.model)
model.generator.model.load_state_dict(ckpt['lora_state_dict'], strict=False)
action_encoder = ActionEncoderV2(out_dim=1536).to(device=device, dtype=torch.bfloat16)
action_encoder.load_state_dict(ckpt['action_encoder_state_dict'])
A_MEAN, A_STD = ckpt['action_mean'].to(device), ckpt['action_std'].to(device)
COND = model.text_encoder(text_prompts=['a robot arm pushing objects on a table'])

env = lmdb.open(LMDB_PATH, readonly=True, lock=False)
LAT_SHAPE = get_array_shape_from_lmdb(env,'latents')
F = LAT_SHAPE[1]; N_CTX = config.num_frame_per_block
VM = torch.eye(4, device=device, dtype=torch.bfloat16).view(1,1,4,4).repeat(1,F,1,1)
KS = torch.tensor([[.5,0,.5],[0,.5,.5],[0,0,1]], device=device, dtype=torch.bfloat16).view(1,1,3,3).repeat(1,F,1,1)
model.scheduler.set_timesteps(N_STEPS); SCHEDULE = model.scheduler.timesteps.to(device)
print(f'model spreman za {time.time()-t0:.0f}s | checkpoint step={ckpt["step"]} rank={rank} | '
      f'{LAT_SHAPE[0]} test scena')

In [ ]:
# ==== funkcije ====
def decode(lat):
    x = model.vae.decode_to_pixel(lat.to(device))
    return ((x.float().clamp(-1,1)+1)/2*255).byte()[0].permute(0,2,3,1).cpu().numpy()

def embed(prev_dir, cur_dir):
    """frejm 0 = nule (kao u treningu), 1..3 = prethodna akcija, 4..7 = komandovana akcija"""
    apl = np.zeros((F,16), dtype=np.float32)
    for i in range(1, N_CTX):
        dx,dy = DIRS[prev_dir]; apl[i] = np.tile([dx,dy,0.5,0.25],4)
    for i in range(N_CTX, F):
        dx,dy = DIRS[cur_dir];  apl[i] = np.tile([dx,dy,0.5,0.25],4)
    a = torch.tensor(apl, device=device).unsqueeze(0)
    an = (a - A_MEAN)/A_STD; an[:,0,:] = 0.0
    assert an.abs().max().item() < 20, 'akcija izvan distribucije'
    return action_encoder(an.to(torch.bfloat16))

def step_block(context, prev_dir, cur_dir):
    """jedan blok: kontekst (4 frejma) + akcija -> sledeca 4 frejma"""
    window = torch.cat([context, torch.randn_like(context)], dim=1)
    ae = embed(prev_dir, cur_dir)
    clean = torch.cat([context, context], dim=1)
    for i, t_val in enumerate(SCHEDULE):
        ts = torch.zeros((1,F), device=device, dtype=torch.bfloat16); ts[:,N_CTX:] = t_val.item()
        window[:,:N_CTX] = context
        _, x0 = model.generator(noisy_image_or_video=window, conditional_dict=COND, timestep=ts,
                                clean_x=clean, aug_t=None, viewmats=VM, Ks=KS, action_embed=ae)
        x0 = x0.float().clamp(-6,6)
        if i == len(SCHEDULE)-1:
            window[:,N_CTX:] = x0[:,N_CTX:].to(torch.bfloat16)
        else:
            sn = float(model.scheduler.sigmas[i+1])
            window[:,N_CTX:] = ((1-sn)*x0 + sn*torch.randn_like(x0))[:,N_CTX:].to(torch.bfloat16)
    return window[:,N_CTX:].clone()

def strip(frames, scale=3):
    g = np.concatenate(list(frames), axis=1)
    return Image.fromarray(g).resize((g.shape[1]*scale, g.shape[0]*scale), Image.NEAREST)
print('funkcije OK')

In [ ]:
# ==== INTERAKTIVNI DEMO ====
SCENE_IDX = 3     # promijeni pa ponovo pokreni ovu ćeliju za drugu scenu

state = {}
def reset(_=None):
    lat = retrieve_row_from_lmdb(env,'latents',np.float16,SCENE_IDX,shape=LAT_SHAPE[1:]).astype(np.float32)
    real = torch.from_numpy(lat).to(device=device, dtype=torch.bfloat16).unsqueeze(0)
    state['context'] = real[:,:N_CTX].clone()
    state['latents'] = [real[:,:N_CTX].clone()]
    state['prev']    = 'still'
    state['history'] = []
    redraw('start (pravi kontekst)', None)

out = W.Output()
def redraw(msg, dt):
    full = torch.cat(state['latents'], dim=1)
    frames = decode(full)
    with out:
        clear_output(wait=True)
        print(f"{msg}" + (f"   [{dt:.2f}s]" if dt else ""))
        print(f"niz: {' -> '.join(state['history']) if state['history'] else '(prazno)'}")
        print(f"{full.shape[1]} latentnih / {len(frames)} piksel-frejmova   "
              f"(prvih {1+4*(N_CTX-1)} = pravi kontekst, ostalo generisano)")
        display(strip(frames))

def go(direction):
    def handler(_):
        for b in buttons: b.disabled = True
        try:
            t0 = time.time()
            blk = step_block(state['context'], state['prev'], direction)
            state['latents'].append(blk); state['context'] = blk
            state['prev'] = direction; state['history'].append(direction)
            redraw(f'akcija: {direction}', time.time()-t0)
        finally:
            for b in buttons: b.disabled = False
    return handler

# obicna slova umjesto Unicode strelica -- neki fontovi nemaju te glifove
L = W.Layout(width='110px')
b_up    = W.Button(description='GORE',   button_style='primary', layout=L)
b_down  = W.Button(description='DOLJE',  button_style='primary', layout=L)
b_left  = W.Button(description='LIJEVO', button_style='primary', layout=L)
b_right = W.Button(description='DESNO',  button_style='primary', layout=L)
b_still = W.Button(description='MIRUJ',  button_style='info',    layout=L)
b_reset = W.Button(description='RESET',  button_style='warning', layout=L)
buttons = [b_up,b_down,b_left,b_right,b_still,b_reset]
for b,d in [(b_up,'up'),(b_down,'down'),(b_left,'left'),(b_right,'right'),(b_still,'still')]:
    b.on_click(go(d))
b_reset.on_click(reset)

display(W.VBox([W.HBox([W.Label(''), b_up, W.Label('')]),
                W.HBox([b_left, b_still, b_right]),
                W.HBox([W.Label(''), b_down, W.Label('')]),
                W.HBox([b_reset]), out]))
reset()

---
### Snimanje niza kao mp4 (opciono)
Pokreni poslije nekoliko akcija.

In [ ]:
import imageio
frames = decode(torch.cat(state['latents'], dim=1))
name = '_'.join(state['history']) or 'ctx'
path = f'/home/mls10/logs/interactive_{name}.mp4'
imageio.mimsave(path, list(frames), fps=4, macro_block_size=1)
print('snimljeno:', path, '| desni klik u file browseru -> Download (mp4 se ne otvara klikom)')